In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    _p = _root / "paths.py"
    if _p.exists() and "SHARED_ROOT" in _p.read_text(encoding="utf-8", errors="ignore"):
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find After_PT_Removal/shared/paths.py — set Jupyter cwd to After_PT_Removal, shared/, or shared/notebooks/."
    )
import paths


In [1]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm.auto import tqdm
tqdm.pandas()

/data/healthy-ml/scratch/haoran/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/data/healthy-ml/scratch/haoran/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv('/data/healthy-ml/scratch/haoran/High_Low_Irr_Data_Source_.csv')
df['High'] = df['High'].apply(lambda x: eval(x))
df['Low+Irr'] = df['Low+Irr'].apply(lambda x: eval(x))

In [3]:
MODEL_NAME = "gpt2-xl"         
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
mod = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE).eval()
    
def string_perplexity(model, tokenizer, text):
    enc = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        loss = model(**enc, labels=enc["input_ids"]).loss
    return np.exp(loss.item())          # perplexity = e^{mean NLL}

def list_stats(text_list):
    ppls = [string_perplexity(mod, tok, s) for s in text_list]
    return ppls

In [4]:
high_perplexity = []
irr_perplexity = []

In [ ]:
for idx, row in tqdm(df.iterrows()):
    high_perplexity.append(list_stats(row['High']))
    irr_perplexity.append(list_stats(row['Low+Irr']))

0it [00:00, ?it/s]

In [ ]:
df['High_perplexity'] = pd.Series(high_perplexity)
df['Irr_perplexity'] = pd.Series(irr_perplexity)

In [ ]:
def compute_stats(x):
    all_high = [j for i in x['High_perplexity'].values for j in i]
    all_irr = [j for i in x['Irr_perplexity'].values for j in i]
    
    return pd.Series({
        'High_mean': np.mean(all_high),
        'High_std': np.std(all_high),
        'Irr_mean': np.mean(all_irr),
        'Irr_std': np.std(all_irr)
    })

In [14]:
final_df = df.groupby(['data_source_df3']).apply(compute_stats)
final_df.loc['Overall'] = compute_stats(df)
final_df.round(1)

/tmp/ipykernel_4065488/171899729.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_df = df.groupby(['data_source_df3']).apply(compute_stats)


,High_mean,High_std,Irr_mean,Irr_std
data_source_df3,,,,
jama,51.6,69.3,68.2,92.4
medbullets,46.5,51.1,48.3,65.8
medxpert,41.4,43.8,52.3,71.0
mmlu,46.4,56.3,58.7,70.4
Overall,48.7,62.0,61.0,82.9


## keywords

In [ ]:
import re, string, math, collections
from typing import List, Dict, Tuple
from sklearn.feature_extraction.text import CountVectorizer

_punc_table = str.maketrans("", "", string.punctuation)

def _clean(text: str) -> str:
    """Lower‑case, strip punctuation/digits, collapse whitespace."""
    text = text.lower()
    text = re.sub(r"\d+", " ", text)                # remove digits
    text = text.translate(_punc_table)              # strip punctuation
    return re.sub(r"\s+", " ", text).strip()

def _term_counts(docs: List[str],
                 stop_words="english") -> Tuple[Dict[str, int], int]:
    """
    Returns:
        counts : dict[word] -> corpus frequency
        total  : total #tokens
    """
    vec = CountVectorizer(stop_words=stop_words, preprocessor=_clean)
    X   = vec.fit_transform(docs)
    vocab = vec.get_feature_names_out()
    counts = X.sum(axis=0).A1
    return dict(zip(vocab, counts)), int(counts.sum())

def _log_odds_ratio(word: str,
                    C1: Dict[str, int],
                    C2: Dict[str, int],
                    N1: int,
                    N2: int,
                    alpha: float = 1.0) -> float:
    """
    Add‑alpha smoothed log‑odds (Monroe et al. 2008 style).
    Positive  -> enriched in corpus 1
    Negative  -> enriched in corpus 2
    """
    # add‑alpha smoothing
    n1 = C1.get(word, 0) + alpha
    n2 = C2.get(word, 0) + alpha
    t1 = N1 + alpha * len(set(C1) | set(C2))
    t2 = N2 + alpha * len(set(C1) | set(C2))
    return math.log(n1 / (t1 - n1)) - math.log(n2 / (t2 - n2))

In [ ]:
def compute_keyword_diff(lst1: List[str],
                         lst2: List[str],
                         top_n: int = 20,
                         stop_words: str = "english",
                         alpha: float = 1.0) -> Dict[str, List[Tuple]]:
    C1, N1 = _term_counts(lst1, stop_words)
    C2, N2 = _term_counts(lst2, stop_words)

    vocab = set(C1) | set(C2)
    scores = {
        w: _log_odds_ratio(w, C1, C2, N1, N2, alpha=alpha)
        for w in vocab
    }

    top_lst1 = sorted(
        ((w, C1.get(w, 0), C2.get(w, 0), scores[w]) for w in vocab if scores[w] > 0),
        key=lambda t: t[3],
        reverse=True
    )[:top_n]

    top_lst2 = sorted(
        ((w, C1.get(w, 0), C2.get(w, 0), scores[w]) for w in vocab if scores[w] < 0),
        key=lambda t: t[3]
    )[:top_n]

    return {
        "top_lst1": top_lst1,
        "top_lst2": top_lst2,
        "scores": scores,
    }

In [ ]:
def explode(x):
    return [j for i in x for j in i]

In [ ]:
def apply_fn(x):
    res = compute_keyword_diff(explode(x['High'].values), explode(x['Low+Irr'].values), top_n =10)
    return pd.Series({
        'High_keywords': [i[0] for i in res['top_lst1']],
        'Low_keywords': [i[0] for i in res['top_lst2']]
    })

In [15]:
pd.set_option('display.max_colwidth', 200)

final_df2 = df.groupby(['data_source_df3']).apply(apply_fn)
final_df2.loc['Overall'] = apply_fn(df)
final_df2

/tmp/ipykernel_4065488/1912045556.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_df2 = df.groupby(['data_source_df3']).apply(apply_fn)


,High_keywords,Low_keywords
data_source_df3,,
jama,"[foveal, hyperreflective, spots, girl, progressively, hypopyon, cytoplasm, punctate, ventricular, man]","[swab, travel, procedures, order, allergic, noncontributory, pertinent, digital, empirically, animals]"
medbullets,"[extremity, increased, meql, developed, flexion, bright, right, lateral, poor, progressively]","[metformin, sexually, active, clear, medications, uncomplicated, known, cervical, nonfocal, albuterol]"
medxpert,"[progressive, severe, levels, low, urea, spine, nitrogen, labor, iliac, mmoll]","[trauma, allergies, appropriately, murmurs, vitamin, beers, personal, resuscitation, ordered, taking]"
mmlu,"[days, emergency, department, shortness, difficulty, leukocyte, ul, urine, white, previously]","[unremarkable, currently, smoke, controlled, illicit, bmi, illness, oral, kgm, weighs]"
Overall,"[spots, foveal, hyperreflective, progressive, hypopyon, watery, cytoplasm, punctate, particularly, wrist]","[organomegaly, smoke, walks, comfortable, noncontributory, personal, nonfocal, antihypertensive, weekends, case]"
